# Google Deepmind Embedding Gemma 2

Google DeepMind EmbeddingGemma2

Download EmbeddingGemma 2, embed the documents in `.data/bronze/` (PDFs, images, videos,
audio and text), store them in Milvus Lite under `.data/gold/db/`, and search them.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (10, 5)

print(f"pandas {pd.__version__} | numpy {np.__version__}")

pandas 3.1.1 | numpy 2.5.4


In [ ]:
from pathlib import Path

# Works whether the kernel starts in notebooks/ or at the repository root.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir()),
    Path.cwd(),
)
DATA_DIR = PROJECT_ROOT / ".data"

print("project root:", PROJECT_ROOT)
sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()) if DATA_DIR.is_dir() else []

## Download from HF

In [3]:
# Download the model and move it to .models
from huggingface_hub import snapshot_download

MODEL_ID = "google/embeddinggemma-2"
MODELS_DIR = PROJECT_ROOT / ".models"
MODEL_DIR = MODELS_DIR / MODEL_ID.split("/")[-1]  # .models/embeddinggemma-2

# Download straight into .models/ rather than into the shared Hugging Face cache
# and moving it out: moving files out of the cache leaves it broken, and copying
# them keeps a second 1.5 GB on disk. The repo is public, so no token is needed.
# Re-running is cheap -- files already in MODEL_DIR are skipped.
snapshot_download(repo_id=MODEL_ID, local_dir=MODEL_DIR)

# What landed on disk (.cache/ is Hugging Face's own download bookkeeping).
files = sorted(
    p for p in MODEL_DIR.rglob("*")
    if p.is_file() and ".cache" not in p.relative_to(MODEL_DIR).parts
)
print("model saved to:", MODEL_DIR.relative_to(PROJECT_ROOT))
pd.DataFrame({
    "file": [str(p.relative_to(MODEL_DIR)) for p in files],
    "size_mb": [round(p.stat().st_size / 1e6, 1) for p in files],
})

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

model saved to: .models/embeddinggemma-2


,file,size_mb
0,.gitattributes,0.0
1,1_Pooling/config.json,0.0
2,2_Normalize/config.json,0.0
3,README.md,0.0
4,chat_template.jinja,0.0
5,config.json,0.0
6,config_sentence_transformers.json,0.0
7,model.safetensors,1488.9
8,modules.json,0.0
9,preprocessor_config.json,0.0


## Generate Embeddings

In [4]:
# Load EmbeddingGemma 2 from .models/ (no network needed)
import torch
from sentence_transformers import SentenceTransformer

MODEL_DIR = PROJECT_ROOT / ".models" / "embeddinggemma-2"  # downloaded above

DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
# Never float16: the model silently returns NaNs (see the model card). bfloat16 on
# CUDA; float32 elsewhere -- on Apple silicon bfloat16 only saved seconds here.
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32

model = SentenceTransformer(str(MODEL_DIR), device=DEVICE, model_kwargs={"dtype": DTYPE})
# The tokenizer sets no length limit, so cap inputs at the model's 8K context.
model.max_seq_length = 8192

print(f"{MODEL_DIR.name} on {DEVICE} ({DTYPE}) | {model.get_embedding_dimension()}-d | modalities: {model.modalities}")

W1109 13:11:04.766000 54245 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]

embeddinggemma-2 on mps (torch.float32) | 768-d | modalities: ['text', 'image', 'audio', 'video', 'message']


In [5]:
# How each document type becomes model input. Every reader yields (metadata, input)
# pairs, one pair per embedding. `unit` says what the embedding covers; the other
# fields say where it sits in the document, so a search can point back at it:
#   image -> "image":   the whole file
#   pdf   -> "page":    the rendered page *and* its text layer, embedded together
#   video -> "segment": SEGMENT_S seconds -- frames at VIDEO_FPS plus that stretch of audio
#            "frame":   each of those frames on its own, to pinpoint a match in a segment
#   audio -> "segment": SEGMENT_S seconds
#   text  -> "chunk":   TEXT_CHUNK_TOKENS tokens, with the lines they span
import math

import av
import pypdfium2 as pdfium
from PIL import Image, ImageOps

PAGE_LONG_SIDE_PX = 1600  # render size; the model then resizes to its own token budget
SEGMENT_S = 30            # 30 frames x 140 tokens + 30 s of audio (750 tokens) fits the 8K context
VIDEO_FPS = 1             # the model's default frame rate
VIDEO_FRAME_WIDTH = 960   # decode size; plenty for 140 tokens a frame
VIDEO_WITH_AUDIO = True   # embed each video segment's soundtrack together with its frames
AUDIO_RATE = 16_000       # the audio encoder expects mono 16 kHz
TEXT_CHUNK_TOKENS = 1024  # text files: tokens per embedding (the model reads up to 8K)


def image_units(path):
    with Image.open(path) as img:
        yield {"unit": "image"}, ImageOps.exif_transpose(img)  # respect camera rotation


def pdf_units(path):
    pdf = pdfium.PdfDocument(path)
    try:
        # Model card format for documents: "title: {title} | text: {content}", "none" if untitled.
        title = (pdf.get_metadata_dict().get("Title") or "").strip() or "none"
        for i in range(len(pdf)):
            page = pdf[i]
            width, height = page.get_size()
            image = page.render(scale=PAGE_LONG_SIDE_PX / max(width, height)).to_pil()
            textpage = page.get_textpage()
            text = " ".join(textpage.get_text_bounded().split())
            textpage.close()
            page.close()
            # The image goes first, so the 8K cap can only ever cut text.
            yield {"unit": "page", "page": i + 1}, {"text": f"title: {title} | text: <|image|> {text}".rstrip(), "image": image}
    finally:
        pdf.close()


def text_units(path):
    text = path.read_text(encoding="utf-8", errors="replace")
    offsets = model.tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
    line, counted_to = 1, 0
    for i in range(0, len(offsets), TEXT_CHUNK_TOKENS):
        start, end = offsets[i][0], offsets[min(i + TEXT_CHUNK_TOKENS, len(offsets)) - 1][1]
        line += text.count("\n", counted_to, start)  # the line `start` is on
        counted_to = start
        last_line = line + text.count("\n", start, max(start, end - 1))
        yield ({"unit": "chunk", "line_start": line, "line_end": last_line},
               f"title: none | text: {' '.join(text[start:end].split())}")


def load_audio(path):
    """The whole soundtrack as mono 16 kHz float32, and the time it starts at."""
    with av.open(str(path)) as container:
        if not container.streams.audio:
            return None, 0.0
        resampler = av.AudioResampler(format="flt", layout="mono", rate=AUDIO_RATE)
        chunks, start = [], None
        for frame in container.decode(audio=0):
            start = frame.time if start is None else start
            chunks += [f.to_ndarray().ravel() for f in resampler.resample(frame)]
        chunks += [f.to_ndarray().ravel() for f in resampler.resample(None)]  # flush
    return (np.concatenate(chunks), start or 0.0) if chunks else (None, 0.0)


def audio_units(path):
    audio, start = load_audio(path)
    if audio is None:
        raise ValueError(f"{path.name} has no audio track")
    step = SEGMENT_S * AUDIO_RATE
    for i in range(0, len(audio), step):
        clip = audio[i:i + step]
        if i and len(clip) < AUDIO_RATE:
            break  # drop a sub-second tail: too short to mean anything on its own
        meta = {"unit": "segment", "start_s": round(start + i / AUDIO_RATE, 2),
                "end_s": round(start + (i + len(clip)) / AUDIO_RATE, 2)}
        yield meta, {"audio": {"array": clip, "sampling_rate": AUDIO_RATE}}


def video_units(path):
    audio, audio_start = load_audio(path) if VIDEO_WITH_AUDIO else (None, 0.0)

    def segment(frames, times, numbers):
        start, end = times[0], times[0] + len(frames) / VIDEO_FPS
        inputs = {"video": frames}
        if audio is not None:
            a, b = (max(0, int((t - audio_start) * AUDIO_RATE)) for t in (start, end))
            if audio[a:b].size:
                inputs["audio"] = {"array": audio[a:b], "sampling_rate": AUDIO_RATE}
        yield {"unit": "segment", "start_s": round(start, 2), "end_s": round(end, 2)}, inputs
        for image, t, n in zip(frames, times, numbers):
            yield {"unit": "frame", "frame": n, "time_s": round(t, 2)}, image

    with av.open(str(path)) as container:
        stream = container.streams.video[0]
        stream.thread_type = "AUTO"
        width = min(VIDEO_FRAME_WIDTH, stream.width)
        height = round(stream.height * width / stream.width / 2) * 2
        frames, times, numbers, next_t = [], [], [], -math.inf
        for n, frame in enumerate(container.decode(stream)):  # n: the frame's number in the file
            if frame.time is None or frame.time < next_t:
                continue
            next_t = (math.floor(frame.time * VIDEO_FPS) + 1) / VIDEO_FPS  # next whole tick
            frames.append(frame.to_image(width=width, height=height))
            times.append(frame.time)
            numbers.append(n)
            if len(frames) == SEGMENT_S * VIDEO_FPS:
                yield from segment(frames, times, numbers)
                frames, times, numbers = [], [], []
        if frames:
            yield from segment(frames, times, numbers)


# Every supported document type: its reader, and the file extensions that mean it.
DOC_TYPES = {
    "image": (image_units, {".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp", ".tif", ".tiff"}),
    "pdf": (pdf_units, {".pdf"}),
    "video": (video_units, {".mp4", ".mov", ".m4v", ".mkv", ".avi", ".webm"}),
    "audio": (audio_units, {".mp3", ".wav", ".flac", ".ogg", ".m4a", ".aac", ".opus"}),
    "text": (text_units, {".txt", ".md", ".markdown", ".rst"}),
}
TYPE_BY_EXTENSION = {ext: doc_type for doc_type, (_, exts) in DOC_TYPES.items() for ext in exts}

In [6]:
# generate_embeddings(...): one or many documents, in any mix of formats, in -> embeddings out
import logging
from itertools import islice

from tqdm.auto import tqdm

UNITS_PER_CALL = 8  # pages / segments / chunks held in memory and encoded together
BATCH_SIZE = 4      # forward-pass batch; lower it if the GPU runs out of memory
POSITION_COLUMNS = ["unit", "page", "line_start", "line_end", "start_s", "end_s", "frame", "time_s"]
log = logging.getLogger("generate_embeddings")


def generate_embeddings(doc_type, path=None, *, skip_errors: bool = False) -> pd.DataFrame:
    """Embed one or more documents, in any mix of formats, with EmbeddingGemma 2.

        generate_embeddings("pdf", "a.pdf")                                  # one document
        generate_embeddings("pdf", ["a.pdf", "b.pdf"])                       # several of one type
        generate_embeddings({"pdf": ["a.pdf", "b.pdf"], "video": "c.mp4"})   # several types at once
        generate_embeddings([("pdf", "a.pdf"), ("image", "d.png")])          # the same, as pairs
        generate_embeddings("auto", ["a.pdf", "c.mp4", "d.png"])             # type from each extension

    Types are "image", "pdf", "video", "audio", "text" or "auto"; plurals ("images") and
    file extensions ("png") work too. Locations may be relative to the project root.

    By default, a missing file or unsupported type stops the call before anything is
    embedded, and a file that fails to read stops it there. With skip_errors=True those
    documents are skipped instead and listed in one logged warning.

    Returns one row per embedding with the document's `path` and `doc_type`, what the
    embedding covers (`unit`: image, page, segment, frame or chunk), where that is in the
    document (`page`; `line_start`/`line_end`; `start_s`/`end_s`; `frame`/`time_s`), and a
    768-d unit-length `embedding`. As a plain (n, 768) array: np.vstack(result["embedding"]).
    """
    # Check every document before embedding any, so a typo in the last one fails fast.
    documents, problems = [], []
    for given_type, given_path in _as_pairs(doc_type, path):
        try:
            documents.append((str(given_path), *_resolve(given_type, given_path)))
        except (ValueError, FileNotFoundError) as err:
            problems.append(f"{given_path}: {err}")
    if problems and not skip_errors:
        raise ValueError(
            f"Can't embed {len(problems)} of {len(problems) + len(documents)} documents:\n  " + "\n  ".join(problems)
        )

    results = []
    for given, key, file in tqdm(documents, desc="embedding", unit="doc", disable=len(documents) < 2):
        try:
            results.append(_embed_document(key, file).assign(path=given, doc_type=key))
        except Exception as err:
            if not skip_errors:
                err.add_note(f"while embedding {given}")
                raise
            problems.append(f"{given}: {err!r}")
    if problems:
        log.warning("Skipped %d of %d documents:\n  %s",
                    len(problems), len(problems) + len(results), "\n  ".join(problems))

    if not results:
        return pd.DataFrame(columns=["path", "doc_type", "unit", "embedding"])
    out = pd.concat(results, ignore_index=True)
    position = [c for c in POSITION_COLUMNS if c in out]
    integers = {c: "Int64" for c in ("page", "line_start", "line_end", "frame") if c in out}
    return out[["path", "doc_type", *position, "embedding"]].astype(integers)


# --- helpers -----------------------------------------------------------------

def _as_pairs(doc_type, path):
    """Every accepted call shape -> a flat list of (doc_type, location)."""
    if path is not None:
        pairs = [(doc_type, path)]
    elif isinstance(doc_type, dict):
        pairs = doc_type.items()
    elif isinstance(doc_type, (str, Path)):
        raise TypeError(f"No location given for {doc_type!r}: call generate_embeddings(doc_type, path).")
    else:
        pairs = doc_type
    flat = []
    for pair in pairs:
        if isinstance(pair, (str, Path)) or len(pair) != 2:
            raise TypeError(
                f"Expected (doc_type, location) pairs, got {pair!r}. "
                "For a list of files, use generate_embeddings('auto', files)."
            )
        doc_type, locations = pair
        flat += [(doc_type, loc) for loc in ([locations] if isinstance(locations, (str, Path)) else locations)]
    return flat


def _canonical_type(name):
    """'images', 'png', '.PNG', 'pdf', ... -> 'image', 'image', 'image', 'pdf'; None if unsupported."""
    key = str(name).lower().strip().lstrip(".")
    key = TYPE_BY_EXTENSION.get("." + key, key.removesuffix("s"))
    return key if key in DOC_TYPES else None


def _resolve(doc_type, location):
    """One (doc_type, location) as given -> (supported type, existing file), or raise saying why not."""
    location = Path(location).expanduser()
    auto = str(doc_type).strip().lower() == "auto"
    name = location.suffix if auto else str(doc_type)
    key = _canonical_type(name) if name else None
    if key is None:
        if auto:
            raise ValueError(f"unsupported file type {name!r}" if name else "no file extension to tell its type from")
        raise ValueError(f"unsupported document type {name!r} (use {', '.join(DOC_TYPES)}, a file extension, or 'auto')")
    if not location.exists() and not location.is_absolute():
        location = PROJECT_ROOT / location  # the kernel usually runs from notebooks/
    if not location.is_file():
        raise FileNotFoundError("not a file" if location.exists() else "no such file")
    return key, location


def _embed_document(doc_type, path):
    """One document -> a frame of its position columns plus `embedding`."""
    reader, _ = DOC_TYPES[doc_type]
    units, metas, vectors = reader(path), [], []
    while chunk := list(islice(units, UNITS_PER_CALL)):
        chunk_metas, inputs = zip(*chunk)
        vectors.append(model.encode(
            list(inputs),
            batch_size=BATCH_SIZE,
            # Video frames are already sampled at VIDEO_FPS -- keep every one of them.
            processing_kwargs={"video": {"do_sample_frames": False}},
            show_progress_bar=False,
        ))
        metas += chunk_metas
    if not metas:
        raise ValueError("nothing to embed")
    embeddings = np.vstack(vectors).astype(np.float32)
    embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)  # exact unit length, whatever DTYPE was
    return pd.DataFrame(metas).assign(embedding=list(embeddings))

In [7]:
# Examples: document type(s) + location(s) in, embeddings out

# One document
flyer = generate_embeddings("image", ".data/bronze/imgs/squarespace-flyer-example.jpg")
print("one image  ->", np.vstack(flyer["embedding"]).shape)

# Several documents in several formats, in one call
mixed = generate_embeddings({
    "pdf": [
        ".data/bronze/docs/rema1000_v1.pdf",
        ".data/bronze/docs/power_3826_dynamic.pdf",
        ".data/bronze/docs/Handelsbetingelser_føtex.dk_-_1.48.pdf"
    ],
    "images": [
        ".data/bronze/imgs/squarespace-flyer-example.jpg",
        ".data/bronze/imgs/05f3e672-953d-4ef8-915b-49a902171dbd.png",
        ".data/bronze/imgs/102df733-d50b-46b3-ac95-eb628953b2cc.png",
        ".data/bronze/imgs/760aed97-64b2-491f-a646-32c19e390e28.png",
        ".data/bronze/imgs/amex.jpg"
    ],
    "video": [
        ".data/bronze/videos/osms-demo.mp4"
    ],
})
print("mixed call ->", np.vstack(mixed["embedding"]).shape)
mixed.groupby(["path", "doc_type", "unit"], sort=False).size().rename("embeddings").reset_index()

one image  -> (1, 768)


embedding:   0%|          | 0/9 [00:00<?, ?doc/s]

mixed call -> (285, 768)


,path,doc_type,unit,embeddings
0,.data/bronze/docs/rema1000_v1.pdf,pdf,page,30
1,.data/bronze/docs/power_3826_dynamic.pdf,pdf,page,47
2,.data/bronze/docs/Handelsbetingelser_føtex.dk_...,pdf,page,17
3,.data/bronze/imgs/squarespace-flyer-example.jpg,image,image,1
4,.data/bronze/imgs/05f3e672-953d-4ef8-915b-49a9...,image,image,1
5,.data/bronze/imgs/102df733-d50b-46b3-ac95-eb62...,image,image,1
6,.data/bronze/imgs/760aed97-64b2-491f-a646-32c1...,image,image,1
7,.data/bronze/imgs/amex.jpg,image,image,1
8,.data/bronze/videos/osms-demo.mp4,video,segment,6
9,.data/bronze/videos/osms-demo.mp4,video,frame,180


## Vector Database

In [ ]:
# Milvus Lite: the embeddings live in .data/gold/db/, searchable by vector and filterable
# by type, file and position. It is the pymilvus API the production Milvus uses too, so
# moving to a server later means changing DB_URI, nothing else.
#
# Only one process can have the database open at a time. To open it from a script or
# another notebook, run close_db() here (or restart the kernel) first.
import json

from milvus_lite.server_manager import server_manager_instance
from pymilvus import DataType, MilvusClient

DB_URI = str(DATA_DIR / "gold" / "db" / "embeddings.db")
COLLECTION = "embeddings"

Path(DB_URI).parent.mkdir(parents=True, exist_ok=True)
if "db" in globals():
    db.close()  # re-running this cell: drop the old connection before making a new one
# pymilvus pings an idle connection every 10 s, more often than the Milvus Lite server
# allows, which floods the cell with gRPC "GOAWAY too_many_pings" errors. Every 5 min is fine.
db = MilvusClient(DB_URI, grpc_options={"grpc.keepalive_time_ms": 300_000})
if not db.has_collection(COLLECTION):
    schema = db.create_schema(auto_id=True, enable_dynamic_field=True)
    schema.add_field("id", DataType.INT64, is_primary=True)
    schema.add_field("embedding", DataType.FLOAT_VECTOR, dim=model.get_embedding_dimension())
    schema.add_field("path", DataType.VARCHAR, max_length=2048)
    schema.add_field("doc_type", DataType.VARCHAR, max_length=16)
    schema.add_field("unit", DataType.VARCHAR, max_length=16)
    # page, line_start/line_end, start_s/end_s and frame/time_s are dynamic fields:
    # each row stores only the ones that apply to it.
    index_params = db.prepare_index_params()
    index_params.add_index("embedding", index_type="FLAT", metric_type="COSINE")  # exact search
    db.create_collection(COLLECTION, schema=schema, index_params=index_params)
db.load_collection(COLLECTION)  # a reopened collection starts out "released" and refuses searches


def store_embeddings(embeddings: pd.DataFrame) -> int:
    """Save rows from generate_embeddings, replacing anything stored before for the same documents."""
    paths = embeddings["path"].unique().tolist()
    if paths:
        db.delete(COLLECTION, filter=f"path in {_literal(paths)}")
    rows = [
        {k: v.item() if hasattr(v, "item") else v for k, v in rec.items() if k != "embedding" and pd.notna(v)}
        | {"embedding": np.asarray(rec["embedding"], dtype=np.float32)}
        for rec in embeddings.to_dict("records")
    ]
    for i in range(0, len(rows), 1000):
        db.insert(COLLECTION, rows[i:i + 1000])
    return len(rows)


def close_db() -> None:
    """Let another process open the database. MilvusClient.close() alone is not enough: the
    Milvus Lite server inside this process keeps the file locked until it is released too."""
    db.close()
    server_manager_instance.release_server(DB_URI)


# Milvus Lite's filter parser takes any character as-is inside quotes and knows only these
# escapes. Not json.dumps: it writes non-ASCII letters such as "ø" as backslash-u codes,
# which the parser rejects.
_ESCAPES = str.maketrans({"\\": "\\\\", '"': '\\"', "\n": "\\n", "\r": "\\r", "\t": "\\t"})


def _literal(value) -> str:
    """A value as a Milvus filter literal. Milvus Lite has no filter placeholders, so values
    go into the expression text, escaped the way its parser expects."""
    if isinstance(value, str):
        return '"' + value.translate(_ESCAPES) + '"'
    if isinstance(value, (list, tuple)):
        return "[" + ", ".join(_literal(v) for v in value) + "]"
    return json.dumps(value)  # numbers

In [9]:
# Embed everything in .data/bronze and store it in the database
bronze_files = sorted(
    p.relative_to(PROJECT_ROOT) for p in (DATA_DIR / "bronze").rglob("*")
    if p.is_file() and not p.name.startswith(".")
)
# "auto": each file's type comes from its extension. skip_errors: unsupported or
# unreadable files are skipped and listed in a warning rather than stopping the run.
embeddings_df = generate_embeddings("auto", bronze_files, skip_errors=True)
stored = store_embeddings(embeddings_df)

in_db = db.query(COLLECTION, filter="", output_fields=["count(*)"])[0]["count(*)"]
print(f"stored {stored} embeddings from {embeddings_df['path'].nunique()} of {len(bronze_files)} files "
      f"-> {Path(DB_URI).relative_to(PROJECT_ROOT)} ({in_db} in the database)")
embeddings_df.groupby(["path", "doc_type", "unit"], sort=False).size().rename("embeddings").reset_index()

embedding:   0%|          | 0/9 [00:00<?, ?doc/s]

stored 285 embeddings from 9 of 9 files -> .data/gold/db/embeddings.db (285 in the database)


,path,doc_type,unit,embeddings
0,.data/bronze/docs/Handelsbetingelser_føtex.dk_...,pdf,page,17
1,.data/bronze/docs/power_3826_dynamic.pdf,pdf,page,47
2,.data/bronze/docs/rema1000_v1.pdf,pdf,page,30
3,.data/bronze/imgs/05f3e672-953d-4ef8-915b-49a9...,image,image,1
4,.data/bronze/imgs/102df733-d50b-46b3-ac95-eb62...,image,image,1
5,.data/bronze/imgs/760aed97-64b2-491f-a646-32c1...,image,image,1
6,.data/bronze/imgs/amex.jpg,image,image,1
7,.data/bronze/imgs/squarespace-flyer-example.jpg,image,image,1
8,.data/bronze/videos/osms-demo.mp4,video,segment,6
9,.data/bronze/videos/osms-demo.mp4,video,frame,180


## Search

In [10]:
# search_embeddings(query): which documents match a query best, and where in them
HIT_FIELDS = ["path", "doc_type", "unit", *POSITION_COLUMNS[1:]]


def search_embeddings(query, *, top_k: int = 10, doc_types=None) -> pd.DataFrame:
    """Search the database for the documents, and the places in them, that best match a query.

        search_embeddings("TV and laptop deals")                            # text
        search_embeddings("TV and laptop deals", doc_types=["pdf", "video"])
        search_embeddings(Path("query.jpg"))                                # a document as the query

    query: text (a str), or a document to find similar content to (a Path, any supported type).
    doc_types: only search these types ("pdf", "images", "video", ...).

    Returns the top_k matches, best first: cosine `score`, `doc_type`, the document's `file`
    name, and where in it the match is -- in words as `location` ("page 14", "frame 1260 at
    0:42 (segment 0:30–1:00)", "1:30–2:00", "lines 120–180", "whole image") and as the
    columns page / line_start, line_end / start_s, end_s / frame, time_s -- plus `path`.
    Videos are ranked by segment (picture and sound together); the frame reported is the
    one inside the winning segment that matches the query best.
    """
    q = _query_vector(query)

    # Rank whole units (pages, images, segments, chunks); frames only pinpoint video hits.
    expr = 'unit != "frame"'
    if doc_types is not None:
        wanted = [_canonical_type(t) for t in ([doc_types] if isinstance(doc_types, str) else doc_types)]
        if None in wanted:
            raise ValueError(f"Unsupported doc_types {doc_types!r} (use {', '.join(DOC_TYPES)})")
        expr += f" and doc_type in {_literal(wanted)}"
    found = db.search(COLLECTION, data=[q], limit=top_k, filter=expr, output_fields=HIT_FIELDS)[0]
    hits = (
        pd.DataFrame([{"score": h["distance"], **h["entity"]} for h in found])
        .reindex(columns=["score", "doc_type", *POSITION_COLUMNS[1:], "path"])
        .astype({"page": "Int64", "line_start": "Int64", "line_end": "Int64", "frame": "Int64"})
    )

    # Pinpoint each video hit: the frame inside its segment that matches the query best.
    for i in hits.index[hits["doc_type"] == "video"]:
        hit = hits.loc[i]
        best = db.search(
            COLLECTION, data=[q], limit=1, output_fields=["frame", "time_s"],
            filter=f'path == {_literal(hit["path"])} and unit == "frame" '
                   f'and time_s >= {hit["start_s"]} and time_s < {hit["end_s"]}',
        )[0]
        if best:
            hits.loc[i, ["frame", "time_s"]] = [best[0]["entity"]["frame"], best[0]["entity"]["time_s"]]

    hits.insert(2, "file", hits["path"].map(lambda p: Path(p).name))
    hits.insert(3, "location", [_describe(hit) for hit in hits.itertuples()])
    return hits.round({"score": 4})


def _query_vector(query):
    """Text -> its search-query embedding; a document (Path) -> the mean of its embeddings."""
    if isinstance(query, Path):
        doc = generate_embeddings("auto", query)
        q = np.vstack(doc.loc[doc["unit"] != "frame", "embedding"]).mean(axis=0)
    else:
        # The model card's retrieval format: queries get "task: search result | query: ".
        q = model.encode(str(query), prompt_name="SearchQuery", show_progress_bar=False)
    return (q / np.linalg.norm(q)).astype(np.float32)


def _clock(seconds):
    minutes, s = divmod(int(round(float(seconds))), 60)  # noqa: RUF046
    h, m = divmod(minutes, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"


def _describe(hit):
    """Where in its document a hit is, in words."""
    if hit.doc_type == "pdf":
        return f"page {hit.page}"
    if hit.doc_type == "image":
        return "whole image"
    if hit.doc_type == "text":
        return f"lines {hit.line_start}–{hit.line_end}"
    span = f"{_clock(hit.start_s)}–{_clock(hit.end_s)}"
    if hit.doc_type == "video" and pd.notna(hit.frame):
        return f"frame {hit.frame} at {_clock(hit.time_s)} (segment {span})"
    return span  # audio, or a video indexed without frames

In [13]:
# Examples: search the database -- every hit says where in its document it matched
from IPython.display import display

# A text query across every format (any language: the model is multilingual)
display(search_embeddings("TV and laptop deals", top_k=5))

# Only videos: the hit names the best-matching frame, and when it is
display(search_embeddings("a software demo", doc_types="video", top_k=3))

# A document as the query: the image itself comes back first, by name
search_embeddings(Path(".data/bronze/imgs/squarespace-flyer-example.jpg"), doc_types="images", top_k=3)

,score,doc_type,file,location,page,line_start,line_end,start_s,end_s,frame,time_s,path
0,0.7154,pdf,power_3826_dynamic.pdf,page 31,31,<NA>,<NA>,NaN,NaN,<NA>,NaN,.data/bronze/docs/power_3826_dynamic.pdf
1,0.7113,pdf,power_3826_dynamic.pdf,page 34,34,<NA>,<NA>,NaN,NaN,<NA>,NaN,.data/bronze/docs/power_3826_dynamic.pdf
2,0.7099,pdf,power_3826_dynamic.pdf,page 1,1,<NA>,<NA>,NaN,NaN,<NA>,NaN,.data/bronze/docs/power_3826_dynamic.pdf
3,0.7066,pdf,power_3826_dynamic.pdf,page 41,41,<NA>,<NA>,NaN,NaN,<NA>,NaN,.data/bronze/docs/power_3826_dynamic.pdf
4,0.7021,pdf,power_3826_dynamic.pdf,page 40,40,<NA>,<NA>,NaN,NaN,<NA>,NaN,.data/bronze/docs/power_3826_dynamic.pdf


,score,doc_type,file,location,page,line_start,line_end,start_s,end_s,frame,time_s,path
0,0.6509,video,osms-demo.mp4,frame 5220 at 2:54 (segment 2:30–3:00),<NA>,<NA>,<NA>,150.0,180.0,5220,174.0,.data/bronze/videos/osms-demo.mp4
1,0.6462,video,osms-demo.mp4,frame 1470 at 0:49 (segment 0:30–1:00),<NA>,<NA>,<NA>,30.0,60.0,1470,49.0,.data/bronze/videos/osms-demo.mp4
2,0.6437,video,osms-demo.mp4,frame 3720 at 2:04 (segment 2:00–2:30),<NA>,<NA>,<NA>,120.0,150.0,3720,124.0,.data/bronze/videos/osms-demo.mp4


,score,doc_type,file,location,page,line_start,line_end,start_s,end_s,frame,time_s,path
0,1.0000,image,squarespace-flyer-example.jpg,whole image,<NA>,<NA>,<NA>,NaN,NaN,<NA>,NaN,.data/bronze/imgs/squarespace-flyer-example.jpg
1,0.6130,image,102df733-d50b-46b3-ac95-eb628953b2cc.png,whole image,<NA>,<NA>,<NA>,NaN,NaN,<NA>,NaN,.data/bronze/imgs/102df733-d50b-46b3-ac95-eb62...
2,0.5809,image,05f3e672-953d-4ef8-915b-49a902171dbd.png,whole image,<NA>,<NA>,<NA>,NaN,NaN,<NA>,NaN,.data/bronze/imgs/05f3e672-953d-4ef8-915b-49a9...
